###  SPARQL Query Sampling Pipeline


In [ ]:

# 1. Import required libraries
import pandas as pd
import requests
from SPARQLWrapper import SPARQLWrapper, JSON
import time
import json

# 2. Set endpoints and parameters
LSQ_ENDPOINT = "http://lsq.aksw.org/sparql"
BATCH_SIZE = 1000
NUM_BATCHES = 10  # Adjust as needed

# 3. Define query template to fetch SELECT queries
QUERY_TEMPLATE = """
PREFIX lsqv: <http://lsq.aksw.org/vocab#>
SELECT DISTINCT ?text
WHERE {{
  ?query lsqv:hasRemoteExec/lsqv:endpoint  ?endpoint .
  ?query lsqv:text ?text .
  ?query lsqv:hasSpin ?spin .
  ?spin a  ?type .
  FILTER regex(str(?text), "^SELECT", "i")
}}
LIMIT {limit}
OFFSET {offset}
"""

In [ ]:
# 4. Fetch queries in batches from LSQ
def fetch_queries(num_batches, batch_size):
    all_queries = []
    sparql = SPARQLWrapper(LSQ_ENDPOINT)
    for i in range(num_batches):
        offset = i * batch_size
        print(f"Batch {i+1}: OFFSET {offset}")
        sparql.setQuery(QUERY_TEMPLATE.format(limit=batch_size, offset=offset))
        sparql.setReturnFormat(JSON)
        try:
            results = sparql.query().convert()
            texts = [res['text']['value'] for res in results['results']['bindings']]
            all_queries.extend(texts)
        except Exception as e:
            print(f"Error: {e}")
        time.sleep(0.3)  # Polite delay
    return pd.DataFrame({'query': all_queries})


In [ ]:
df = fetch_queries(NUM_BATCHES, BATCH_SIZE)
print(f"Fetched queries: {len(df)}")

# 5. Remove duplicate queries
df_unique = df.drop_duplicates(subset='query')
print(f"Unique SELECT queries: {len(df_unique)}")

# 6. Save unique queries to CSV
df_unique.to_csv("unique_select_queries.csv", index=False)
